# 画像認識4：CLIP

[CLIP](https://openai.com/research/clip)とは、ChatGPTでおなじみのopenaiが2021年に発表した物体認識モデルです。   
訓練データに含まれていないデータ (Out-of-Distribution, OOD）でも、Zero-shot、すなわち初見で精度よく物体認識できることで大きなセンセーショナルを巻き起こしました。   
CLIPは文書から画像を生成する[DALL-E2](https://openai.com/dall-e-2)の内部で、テキストと画像を紐づける部分においても利用されています。   
ここでは、自然言語処理モデルであるBERTでも利用したHaggingfaceのAPIを使ってCLIPを試行してみましょう。

- CLIPの論文：Radford, et. al., “Learning Transferable Visual Models From Natural Language Supervision,” arXiv:2103.00020, https://arxiv.org/abs/2103.00020 , 2021
- [HaggingfaceにおけるCLIPのページ](https://huggingface.co/docs/transformers/model_doc/clip)



# Transformersのインストールと画像のダウンロード
以下を実行して、transformersのインストールと、認識対象となる画像をダウンロードしてください。  
**このセルはColaboratoryを起動するたびに必要となります**   
**「ランタイム＞ランタイムのタイプを変更」で「ハードウェアアクセラレータ」をGPUにしてから実行することをお勧めします。**

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install transformers
!wget -P ./img https://park.itc.u-tokyo.ac.jp/yamakata-lab/lecture/mediaproc/mediaproc6/mediaproc6-MobileNetSSD.zip
!unzip img/mediaproc6-MobileNetSSD.zip -d img/

## 1. 画像の読み込み

認識対象とする画像を選んで表示させてみましょう。   
これは何の画像でしょうか？

In [ ]:
from PIL import Image

#image = 'img/bangkok-buildings-cars-708764.jpg' # https://www.pexels.comより取得
image = 'img/adorable-animal-blur-850602.jpg' # https://www.pexels.comより取得
#image = 'img/animals-cats-cute-45170.jpg' # https://www.pexels.comより取得
#image = 'img/aeroplanes-aircraft-airline-163792.jpg' # https://www.pexels.comより取得
#image = "img/1024px-Super_Tuesday_(6814264898).jpg" # https://upload.wikimedia.org/wikipedia/commons/thumb/a/a4/Super_Tuesday_%286814264898%29.jpg/1024px-Super_Tuesday_%286814264898%29.jpg より取得

image = Image.open(image)
image

## 2. モデルのダウンロードと読み込み
Haggingfaceに登録されているCLIPのモデルをダウンロードして読み込みます。


In [ ]:
from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

## 2. 画像とテキストとの類似度計算

この写真に写っている物体は「犬」か「猫」かを分類してみましょう。   
CLIPでは、プロンプトとして"a photo of"が用いられることが多いです。   
よって入力文には以下の２種類が渡されています。   

- "a photo of a cat" （猫の写真）
- "a photo of a dog" （犬の写真）

これと対象画像を入力とし、以下の２つを書き出してみます。

- 画像と各文章の類似度
- 画像が各文章で指定されているラベル（すなわち「猫」と「犬」）に該当する確率

In [ ]:
inputs = processor(text=["a photo of a cat", "a photo of a dog"],
                   images=image,
                   return_tensors="pt",
                   padding=True)

outputs = model(**inputs)
logits_per_image = outputs.logits_per_image
probs = logits_per_image.softmax(dim=1)
print('画像と入力テキストの類似度', logits_per_image.squeeze().tolist())
print('それぞれのテキストに当てはまる確率（物体認識結果）', probs.squeeze().tolist())

この結果、犬である確率が99.96%となります。   
写真や入力文をいろいろと変えて結果を確認してみましょう。